# Create fractals
## Sources:
- [Creating fractals with Python](https://towardsdatascience.com/creating-fractals-with-python-d2b663786da6)
- [A simple introduction to the world of fractals using python](https://godhalakshmi.medium.com/a-simple-introduction-to-the-world-of-fractals-using-python-c8cb859bfd6d)
- [Programming Fractals in Python](https://medium.com/nerd-for-tech/programming-fractals-in-python-d42db4e2ed33)
- [Draw the Mandelbrot Set in Python](https://realpython.com/mandelbrot-set-python/)


## Definitions

## Import Modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
import turtle

# import 3rd-party modules
import numpy as np
from matplotlib import pyplot as plt, cm
from PIL import Image, ImageEnhance
from PIL.ImageColor import getrgb
import cv2
from scipy.interpolate import interp1d

## Viewport

In [ ]:
class Viewport:
    def __init__(self, image, center, width):
        self.image = image
        self.center = center
        self.width = width

    @property
    def height(self):
        return self.width * self.image.height / self.image.width

    @property
    def offset(self):
        return self.center - complex(self.width, self.height) / 2

    def __iter__(self):
        for y in range(self.image.height):
            for x in range(self.image.width):
                yield Pixel(self, x, y)

class Pixel:
    def __init__(self, viewport, x, y):
        self.viewport = viewport
        self.x = x
        self.y = y

    @property
    def color(self):
        return self.viewport.image.getpixel((self.x, self.y))

    @color.setter
    def color(self, value):
        self.viewport.image.putpixel((self.x, self.y), value)

    def __complex__(self):
        return (
            complex(
                self.x * self.viewport.width / self.viewport.image.width,
                self.y * self.viewport.height / self.viewport.image.height,
            )
            + self.viewport.offset
        )

## Mandelbrot Set

In [ ]:
from math import log

class MandelbrotSet:
    def __init__(self, max_iterations, escape_radius=2):
        self.max_iterations = max_iterations
        self.escape_radius = escape_radius

    def __contains__(self, c):
        return self.stability(c) == 1

    def stability(self, c, smooth=False):
        return self.escape_count(c, smooth) / self.max_iterations

    def escape_count(self, c, smooth=False):
        z = 0
        for i in range(self.max_iterations):
            z = z ** 2 + c
            if abs(z) > self.escape_radius:
                if smooth:
                    return i + 1 - log(log(abs(z))) / log(2)
                return i
        return self.max_iterations

## Draw fractals with turtle

In [ ]:
MINIMUM_BRANCH_LENGTH = 5
def build_tree(t, branch_length, shorten_by, angle):
  if branch_length > MINIMUM_BRANCH_LENGTH:
    t.forward(branch_length)
    new_length = branch_length - shorten_by
    t.left(angle)
    build_tree(t, new_length, shorten_by, angle)
    t.right(angle * 2)
    build_tree(t, new_length, shorten_by, angle)
    t.left(angle)
    t.backward(branch_length)
tree = turtle.Turtle()
tree.hideturtle()
tree.setheading(90)
tree.color('green')
build_tree(tree, 50, 5, 30)
turtle.mainloop()

In [ ]:
def koch_curve(t, iterations, length, shortening_factor, angle):
  if iterations == 0:
    t.forward(length)
  else:
    iterations = iterations - 1
    length = length / shortening_factor
    koch_curve(t, iterations, length, shortening_factor, angle)
    t.left(angle)
    koch_curve(t, iterations, length, shortening_factor, angle)
    t.right(angle * 2)
    koch_curve(t, iterations, length, shortening_factor, angle)
    t.left(angle)
    koch_curve(t, iterations, length, shortening_factor, angle)
t = turtle.Turtle()
t.hideturtle()
for i in range(3):
  koch_curve(t, 4, 200, 3, 60)
  t.right(120)
turtle.mainloop()

## Plot fractals with Matplotlib

In [ ]:
# -*- coding: utf-8 -*-
"""
BARNSLEY FERN
By: it's literally monique
"""

# import libraries needed for program
import matplotlib.pyplot as plt 
from random import randint 
  
#initialize list & set first values to 1
x = [0] 
y = [0] 

# create the barnsley fractal by creating the points on the scatterplot
for i in range(0, 50000): 
  
    z = randint(1, 100) 
      
    if z == 1: 
        x.append(0) 
        y.append(0.16*(y[i])) 
         
    if z>= 2 and z<= 86: 
        x.append(0.85*(x[i]) + 0.04*(y[i])) 
        y.append(-0.04*(x[i]) + 0.85*(y[i])+1.6) 
      
    if z>= 87 and z<= 93: 
        x.append(0.2*(x[i]) - 0.26*(y[i])) 
        y.append(0.23*(x[i]) + 0.22*(y[i])+1.6) 
          
    if z>= 94 and z<= 100: 
        x.append(-0.15*(x[i]) + 0.28*(y[i])) 
        y.append(0.26*(x[i]) + 0.24*(y[i])+0.44) 

# make and show the scatterplot
plt.scatter(x, y, s = 0.2, c ='#5dbb63') 
plt.axis("off")
plt.savefig('barnsley_fern.png', dpi=300, bbox_inches='tight')
plt.show()


Now, you can take that matrix of complex numbers and run it through the well-known recursive formula to see which numbers remain stable and which don’t.
Thanks to NumPy’s vectorization, you can pass the matrix as a single parameter, c, and perform the calculations on each element without having to write explicit loops:

To generate the initial set of candidate values, you can take advantage of np.linspace(), which creates evenly spaced numbers in a given range:

In [ ]:
def create_complex_matrix(min_x, max_x, min_y, max_y, pixel_density):
    """
    Function to create a 2d array of complex numbers
    Arguments:
    * min_x, max_x, min_y, max_y: bounds of a rectangle
    * pixel_density: numbers of pixels per unit
    """
    
    # create real numbers, evenly spaced in the following range: number of pixels within bounds of rectangle in x-direction (real axis)
    re = np.linspace(min_x, max_x, int((max_x - min_x) * pixel_density))

    # create imaginary numbers, evenly spaced in the following range: number of pixels within bounds of rectangle in y-direction (imaginary axis)
    im = np.linspace(min_y, max_y, int((max_y - min_y) * pixel_density))

    # return 2d array of complex numbers
    ## make re as row vector (x-direction) by adding an axis in 1st dimension
    ## make im as column vector (y-direction) by adding an axis in 2nd dimension
    return re[np.newaxis, :] + im[:, np.newaxis] * 1j

def is_stable(c, num_iterations, z=0):
    """
    Function to check if a candidate value (complex number c) is stable: 
    stable if the magnitude Zn resulting from the recursive formula exceeds the radius of 2 after n iterations
    This recursive formula gives the Mandelbrot sequence.

    The absolute value of a complex number, a+bi (also called the modulus) is defined as the distance between the origin (0,0) and the point (a,b) in the complex plane.

    By default z = 0 as in a mandelbrot sequence, the first element is always 0
    """
    # # in mandelbrot sequence, the first element is always 0
    # z = 0
    
    # compute sequence over n iterations
    for _ in range(num_iterations):
        z = z ** 2 + c
    
    # check if radius of z doesn't exceed 2
    return abs(z) <= 2


def get_members(c_set, num_iterations):
    """
    Function to check which candidate values in set are stable
    """
    mask = is_stable(c_set, num_iterations)
    return c_set[mask]

In [ ]:
# create matrix of complex numbers (that will be the set of candidate values)
c_set = create_complex_matrix(-2, 0.5, -1.5, 1.5, pixel_density=21)

# check which candidate values in set are stable
members = get_members(c_set, num_iterations=20)

# plot mandelbrot set
plt.scatter(members.real, members.imag, color="black", marker=",", s=1)

# plt.gca() to get current axes, set aspect: ratio of y-unit to x-unit
plt.gca().set_aspect("equal")

# show axis
plt.axis("on")

# show plot
plt.show()

In [ ]:
# create matrix of complex numbers (that will be the set of candidate values)
# set a higher pixel density to get a better resolution
# to zoom in on a particular area, change bounds of complex matrix and number of iterations by 10 or more
c_set = create_complex_matrix(-2, 0.5, -1.5, 1.5, pixel_density=100)

# display data as an image with a binary colormap to plot the boolean mask of stability
plt.imshow(is_stable(c_set, num_iterations=20), cmap="binary")

# plt.gca() to get current axes, set aspect: ratio of y-unit to x-unit
plt.gca().set_aspect("equal")

# show axis
plt.axis("on")

# show plot
plt.show()

In [ ]:
# create matrix of complex numbers (that will be the set of candidate values)
# set a higher pixel density to get a better resolution
# to zoom in on a particular area, change bounds of complex matrix and number of iterations by 10 or more
c_set = create_complex_matrix(-2/2, 0.5/2, -1.5/2, 1.5/2, pixel_density=100*10)

# display data as an image with a binary colormap to plot the boolean mask of stability
plt.imshow(is_stable(c_set, num_iterations=20), cmap="binary")

# plt.gca() to get current axes, set aspect: ratio of y-unit to x-unit
plt.gca().set_aspect("equal")

# show axis
plt.axis("on")

# show plot
plt.show()

## Draw Mandelbrot set with Pillow or OpenCV
the Mandelbrot set appears in black on a white background since Pillow assumes a black background by default.

In [ ]:
# draw mandelbrot set with pillow function

# set size in pixels
width, height = 512, 512

# set bounding boxes as bottom-left and top-right corners
bbx = (-3, -2.5, 2, 2.5)

# set image quality on a scale from 0 to 100
quality = 100
Image.effect_mandelbrot((width, height), bbx, quality = 100).show()

In [ ]:
# create matrix of complex numbers (that will be the set of candidate values)
# set a higher pixel density to get a better resolution
# to zoom in on a particular area, change bounds of complex matrix and number of iterations by 10 or more
c_set = create_complex_matrix(-2, 0.5, -1.5, 1.5, pixel_density=512)

# use of bitwise not operator (~) to invert boolean values of stability matrix -> to make mandelbrot set appears in black on white background
# as pillow assumes black background by default
# then convert array to image
image = Image.fromarray(~is_stable(c_set, num_iterations=20))
image.show()

In [ ]:
# get mandlebrot set (stability matrix, inverse it to get a mandelbrot set in black on white background)
mandelbrot_set = ~is_stable(c_set, num_iterations=20)

# cast set to np.uint8 and multiply by 255
mandelbrot_set = mandelbrot_set.astype(np.uint8)
mandelbrot_set *= 255

# show image
cv2.imshow("fractal", mandelbrot_set)

# wait for user to press any key 
cv2.waitKey(0) 
  
# close all open windows 
cv2.destroyAllWindows()

# workaround to effectively close windows on MacOS
cv2.waitKey(1)

In [ ]:
# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=30)

# check if a complex number is in or not in Mandelbrot set
complex_number = 0.26 + 1j
print(f"Is {complex_number} in Mandelbrot set?", complex_number in mandelbrot_set)
print(f"Is {complex_number} not in Mandelbrot set?", complex_number not in mandelbrot_set)

In [ ]:
# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=30)

# set size in pixels
width, height = 512, 512

# set scale factor between pixel coordinates and world coordinates
scale = 0.0075

# set pixel mode in b&w
BLACK_AND_WHITE = "1"

# create new pillow image (in b&w pixel mode and given size in pixels)
image = Image.new(mode=BLACK_AND_WHITE, size=(width, height))

# iterate over rows
for y in range(height):
    # iterate over columns
    for x in range(width):
        
        # scale and translate each point from pixel to world coords
        c = scale * complex(x - width / 2, height / 2 - y)

        # put a pixel when candidate value c is not in mandlebrot set (so that the mandelbrot set stays black)
        image.putpixel((x, y), c not in mandelbrot_set)

image.show()

In [ ]:
# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=30)

# set candidate value
c = 0.25 + 0j

# check the escape count and stability of candidate value
print(f"Escape count of {c}:", mandelbrot_set.escape_count(c))
print(f"Stability of {c}:", mandelbrot_set.stability(c))

# check if candidate value is in mandelbrot set
print(f"Is {c} in Mandelbrot set?", c in mandelbrot_set)

# set candidate value
c = 0.26 + 0j

# check the escape count and stability of candidate value
print(f"Escape count of {c}:", mandelbrot_set.escape_count(c))
print(f"Stability of {c}:", mandelbrot_set.stability(c))

# check if candidate value is in mandelbrot set
print(f"Is {c} in Mandelbrot set?", c in mandelbrot_set)

In [ ]:
# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=20)

# set size in pixels
width, height = 512, 512

# set scale factor between pixel coordinates and world coordinates
scale = 0.0075

# set pixel mode in luminance
GRAYSCALE = "L"

# create new pillow image (in luminance pixel mode and given size in pixels)
image = Image.new(mode=GRAYSCALE, size=(width, height))

# iterate over rows
for y in range(height):
    # iterate over columns
    for x in range(width):
        
        # scale and translate each point from pixel to world coords
        c = scale * complex(x - width / 2, height / 2 - y)

        # get instability fraction
        instability = 1 - mandelbrot_set.stability(c)

        # put pixel in a level of gray depending on the instability
        # 0 would mean it's stable => put black pixel; 1 it's quickly instable => put white pixel
        image.putpixel((x, y), int(instability * 255))

image.show()

In [ ]:
# try to plot a mandlebrot set with grayscale values of an image
# get image path & read image
img_path = "/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/atomium.jpeg"
img = Image.open(img_path)

# convert rgb image into luminance (i.e. grayscale)
img = img.convert("L")

# convert image to array (# toDo: try to use opencv)
img = np.array(img)

# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=20)

# set size in pixels
width, height = img.shape

# set scale factor between pixel coordinates and world coordinates
scale = 0.05

# set pixel mode in luminance
GRAYSCALE = "L"

# create new pillow image (in luminance pixel mode and given size in pixels)
image = Image.new(mode=GRAYSCALE, size=(width, height))

# iterate over rows
for y in range(height):
    # iterate over columns
    for x in range(width):
        
        # # scale and translate each point from pixel to world coords
        # c = scale * complex(x - width / 2, height / 2 - y)
        c = img[x,y] * scale

        # get instability fraction
        instability = 1 - mandelbrot_set.stability(c)

        # put pixel in a level of gray depending on the instability
        # 0 would mean it's stable => put black pixel; 1 it's quickly instable => put white pixel
        image.putpixel((x, y), int(instability * 255))

image.show()

In [ ]:
# instantiate mandelbrot set class
mandelbrot_set = MandelbrotSet(max_iterations=20, escape_radius=100)

# set size in pixels
width, height = 512, 512

# set scale factor between pixel coordinates and world coordinates
scale = 0.0075

# set pixel mode in luminance
GRAYSCALE = "L"

# create new pillow image (in luminance pixel mode and given size in pixels)
image = Image.new(mode=GRAYSCALE, size=(width, height))

# iterate over rows
for y in range(height):
    # iterate over columns
    for x in range(width):
        
        # scale and translate each point from pixel to world coords
        c = scale * complex(x - width / 2, height / 2 - y)

        # get instability fraction
        instability = 1 - mandelbrot_set.stability(c, smooth=True)

        # put pixel in a level of gray depending on the instability
        # 0 would mean it's stable => put black pixel; 1 it's quickly instable => put white pixel
        image.putpixel((x, y), int(instability * 255))

image.show()

 The Mandelbrot set contains virtually limitless intricate structures that can only be seen under great magnification. Some areas feature spirals and zigzags resembling seahorses, octopuses, or elephants.

In [ ]:
# instantiate mandelbrot set class
# As you zoom in, don’t forget to increase the maximum number of iterations to reveal more detail
mandelbrot_set = MandelbrotSet(max_iterations=256, escape_radius=1000)

# set size in pixels
width, height = 512, 512

# set pixel mode in luminance
GRAYSCALE = "L"

# create new pillow image (in luminance pixel mode and given size in pixels)
image = Image.new(mode=GRAYSCALE, size=(width, height))

# iterate over each pixel in viewport
# The viewport spans 0.002 world units and is centered at -0.7435 + 0.1314j, which is close to a Misiurewicz point that produces a beautiful spiral
for pixel in Viewport(image, center=-0.7435 + 0.1314j, width=0.002):
    
    # convert pixel into a complex number in world units
    c = complex(pixel)

    # get instability fraction
    instability = 1 - mandelbrot_set.stability(c, smooth=True)

    # color pixel
    # put pixel in a level of gray depending on the instability
    # 0 would mean it's stable => put black pixel; 1 it's quickly instable => put white pixel
    pixel.color = int(instability * 255)

# increase brightness by 25%
enhancer = ImageEnhance.Brightness(image)

# show image
enhancer.enhance(1.25).show()
# image.show()

### Paint Mandelbrot set

In [ ]:
def paint(mandelbrot_set, viewport, palette, smooth):
    """
    Function to paint a Mandelbrot set 
    """
    # iterate over each pixel in viewport
    for pixel in viewport:
        
        # get stability
        stability = mandelbrot_set.stability(complex(pixel), smooth)

        # scale scability and clamp it to be able to use it as an index to color palette
        index = int(min(stability * len(palette), len(palette) - 1))

        # color pixel with rgb color corresponding to index
        pixel.color = palette[index % len(palette)]

def denormalize(palette):
    """
    Function to scale fractional values to integer values.
    E.g. (0.13, 0.08, 0.21) will be scaled to (45, 20, 53)
    """
    return [
        tuple(int(channel * 255) for channel in color)
        for color in palette
    ]"

In [ ]:
# get color map
colormap = cm.get_cmap("twilight").colors # reversed color with _r suffix: e.g. twilight_r

# get color palette
palette = denormalize(colormap)

print("number of colors in colormap:", len(colormap))
print("first color in colormap:", colormap[0])
print("first color in palette:", palette[0])

In [ ]:
# instantiate mandelbrot set class
# As you zoom in, don’t forget to increase the maximum number of iterations to reveal more detail
mandelbrot_set = MandelbrotSet(max_iterations=512, escape_radius=1000)

# set size in pixels
width, height = 512, 512

# create new pillow image (in rgb pixel mode and given size in pixels)
image = Image.new(mode="RGB", size=(width, height))

# The viewport spans 0.002 world units and is centered at -0.7435 + 0.1314j, which is close to a Misiurewicz point that produces a beautiful spiral
viewport = Viewport(image, center=-0.7435 + 0.1314j, width=0.002)

# paint mandelbrot set in viewport
paint(mandelbrot_set, viewport, palette, smooth=True)

# show image
image.show()

#### paint with your custom palette

In [ ]:
# create a palette of 100 colors to emphasize the fractal’s edge
# set 50% of palette to exterior
exterior = [(1, 1, 1)] * 50

# set 5% of palette to interior
interior = [(1, 1, 1)] * 5

# set remaining 45% to gray area (in between)
gray_area = [(1 - i / 44,) * 3 for i in range(45)]
palette = denormalize(exterior + gray_area + interior)

In [ ]:
# instantiate mandelbrot set class
# As you zoom in, don’t forget to increase the maximum number of iterations to reveal more detail
mandelbrot_set = MandelbrotSet(max_iterations=20, escape_radius=1000)

# set size in pixels
width, height = 512, 512

# create new pillow image (in rgb pixel mode and given size in pixels)
image = Image.new(mode="RGB", size=(width, height))

#  set the viewport’s center point at -0.75 and its width to 3.5 units to cover the entire fractal
# (At this zoom level, the number of iterations needs to be lower)
viewport = Viewport(image, center=-0.75, width=3.5)

# paint mandelbrot set in viewport
paint(mandelbrot_set, viewport, palette, smooth=True)

# show image
image.show()

In [ ]:
def make_gradient(colors, interpolation="linear"):
    """
    Function to interpolate intermediate color values given a list of colors
    """
    # get list of normalized values between 0 and 1 based on the number of colors
    # X = [i / (len(colors) - 1) for i in range(len(colors))]
    X = np.linspace(0, 1, len(colors)).tolist()

    # get list of Rs, Gs and Bs values from each color
    Y = [[color[i] for color in colors] for i in range(3)]
    channels = [interp1d(X, y, kind=interpolation) for y in Y]
    # return 
    return lambda x: [np.clip(channel(x), 0, 1) for channel in channels]

In [ ]:
# set colors
black = (0, 0, 0)
blue = (0, 0, 1)
maroon = (0.5, 0, 0)
navy = (0, 0, 0.5)
red = (1, 0, 0)

colors = [black, navy, blue, maroon, red, black]

# create lamba function to interpolate intermediate color values
gradient = make_gradient(colors, interpolation="cubic")

# get a gradient color
gradient(0.42)

In [ ]:
# create palette of gradient colors

# set number of colors in palette
num_colors = 256

# get palette of gradient colors
palette = denormalize([
    gradient(i / num_colors) for i in range(num_colors)
])

print("number of colors in palette:", len(palette))
print("sample gradient color:", palette[127])

In [ ]:
# instantiate mandelbrot set class
# As you zoom in, don’t forget to increase the maximum number of iterations to reveal more detail
mandelbrot_set = MandelbrotSet(max_iterations=20, escape_radius=1000)

# set size in pixels
width, height = 512, 512

# create new pillow image (in rgb pixel mode and given size in pixels)
image = Image.new(mode="RGB", size=(width, height))

#  set the viewport’s center point at -0.75 and its width to 3.5 units to cover the entire fractal
# (At this zoom level, the number of iterations needs to be lower)
viewport = Viewport(image, center=-0.75, width=3.5)

# paint mandelbrot set in viewport
paint(mandelbrot_set, viewport, palette, smooth=True)

# show image
image.show()

### better way to manipulate colors
With Hue, Saturation, Brightness (HSB):
- Hue: The angle measured counterclockwise between 0° and 360°
- Saturation: The radius of the cylinder between 0% and 100%
- Brightness: The height of the cylinder between 0% and 100%

In [ ]:
def hsb_to_rgb(hue_degrees: int, saturation: float, brightness: float):
    """
    f
    """
    return getrgb(
        f"hsv({hue_degrees % 360},"
        f"{saturation * 100}%,"
        f"{brightness * 100}%)"
    )

hsb_to_rgb(360, 0.75, 1)

In [ ]:
# instantiate mandelbrot set class
# As you zoom in, don’t forget to increase the maximum number of iterations to reveal more detail
mandelbrot_set = MandelbrotSet(max_iterations=10000, escape_radius=1000)

# set size in pixels
width, height = 1000, 1000

# create new pillow image (in rgb pixel mode and given size in pixels)
image = Image.new(mode="RGB", size=(width, height))

#  set the viewport’s center point at -0.75 and its width to 3.5 units to cover the entire fractal
# (At this zoom level, the number of iterations needs to be lower)
# viewport = Viewport(image, center=-0.75, width=3.5)
for pixel in Viewport(image, center=-0.75, width=0.5):
    stability = mandelbrot_set.stability(complex(pixel), smooth=True)

    # paint interior black (stability 1), else paint exterior with a saturation that fades with distance from the fractal and a hue that follows the HSB cylinder’s angular dimension
    pixel.color = (0, 0, 0) if stability == 1 else hsb_to_rgb(
        hue_degrees=int(stability * 360), # scale stability to 360° degrees
        saturation=stability, # use stability to modulate the saturation
        brightness=1, # set the brightness to 100%
    )

# show image
image.show()

In [ ]:
def mandelbrot(n_rows, n_columns, iterations, cx, cy):
    x_cor = np.linspace(-2, 2, n_rows)
    y_cor = np.linspace(-2, 2, n_columns)
    x_len = len(x_cor)
    y_len = len(y_cor)
    output = np.zeros((x_len,y_len))
    c = complex(cx, cy)
    for i in range(x_len):
        for j in range(y_len):
            z = complex(x_cor[i], y_cor[j])
            count = 0
            for k in range(iterations):
                z = (z * z) + c
                count = count + 1
                if (abs(z) > 4):
                    break
            output[i,j] = count
        print(int((i/x_len)*100),"% completed")

    print(output)
    plt.figure(figsize=(20,20))
    plt.imshow(output.T, cmap='hot')
    plt.axis("off")
    plt.show()

In [ ]:
mandelbrot(1500, 1500, 15, -0.74, 0.15)